<a href="https://colab.research.google.com/github/AarnavSawant/KVCompass/blob/main/analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# KVCompass Analysis Notebook

This notebook is the analysis and visualization half of the original `KVCompass.ipynb`. It assumes the benchmark run artifacts have already been written to Google Drive, then loads the saved summaries and metrics to build tables, recommendations, and plots.


## Shared Analysis Setup

Run this notebook after the benchmark collection notebook has produced the `assignment_*__summary.csv` files and per-run `metrics.json` artifacts under your shared Drive results directory.


In [ ]:
# Shared setup for analysis: mount Drive and install plotting dependencies.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

!python -m pip install --upgrade pip
!pip install pandas matplotlib seaborn numpy


In [ ]:
# Shared values used by the analysis cells.
from pathlib import Path

SHARED_RESULTS_DIR = Path('/content/drive/MyDrive/KVCompass')
SHARED_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Shared results dir:', SHARED_RESULTS_DIR)


## Final Aggregation For The Presentation

After everyone finishes, make sure all of the `assignment_*__summary.csv` files are present under `results/benchmark_eval`, then run the next cells.


In [ ]:
# Load and combine the benchmark assignment summaries.
import json
from pathlib import Path
import pandas as pd

summary_root = SHARED_RESULTS_DIR / 'benchmark_eval'
summary_paths = [summary_root / f'assignment_{i}__summary.csv' for i in range(1, 6) if (summary_root / f'assignment_{i}__summary.csv').exists()]
combined_summary = pd.concat([pd.read_csv(path) for path in summary_paths], ignore_index=True) if summary_paths else pd.DataFrame()
print('Summary files:', [str(path) for path in summary_paths])
display(combined_summary)

if not combined_summary.empty:
    rows = []
    for _, row in combined_summary.iterrows():
        metrics = json.loads(Path(row['metrics_path']).read_text())
        metric_values = []
        for task_metrics in metrics.values():
            if isinstance(task_metrics, dict):
                metric_values.extend(v for v in task_metrics.values() if isinstance(v, (int, float)))
        rows.append({
            'scenario_name': row['scenario_name'],
            'dataset': row['dataset'],
            'data_dir': row['data_dir'],
            'task_prefixes': row['task_prefixes'],
            'method': row['method'],
            'budget': row['budget'],
            'avg_quality': round(sum(metric_values) / len(metric_values), 2) if metric_values else None,
            'avg_latency_seconds': row.get('avg_latency_seconds'),
            'avg_throughput_tokens_per_second': row.get('avg_throughput_tokens_per_second'),
            'peak_gpu_memory_mb': row.get('peak_gpu_memory_mb'),
        })
    leaderboard = pd.DataFrame(rows).sort_values(['scenario_name', 'avg_quality', 'avg_latency_seconds'], ascending=[True, False, True])
    display(leaderboard)
else:
    print('No assignment summaries found yet.')


In [ ]:
# Build a presentation-friendly matrix view.
import pandas as pd

if 'leaderboard' in globals() and not leaderboard.empty:
    leaderboard["scenario_name"] = leaderboard["scenario_name"].replace(
        {'aggregation_4k_baseline': 'aggregation_4k',
         'niah_4k_baseline': 'needle_in_a_haystack_4k',
         'niah_8k_baseline': 'needle_in_a_haystack_8k',
         'qa_4k_baseline': 'question_answering_4k',
         'qa_8k_baseline': 'question_answering_8k',
         'vt_4k_baseline': 'multi_hop_tracing_4k',
         'vt_8k_baseline': 'multi_hop_tracing_8k'
         })
    matrix_view = leaderboard.pivot_table(
        index=['scenario_name', 'data_dir'],
        columns='method',
        values='avg_quality',
        aggfunc='first',
    )
    display(matrix_view)
else:
    print('Leaderboard not ready yet.')


In [ ]:
# Presentation plots.
import matplotlib.pyplot as plt

if 'leaderboard' in globals() and not leaderboard.empty:
    plot_df = leaderboard.copy()
    plot_df['label'] = plot_df['scenario_name'] + ' | ' + plot_df['method'] + ' @ ' + plot_df['budget'].astype(str)
    fig, axes = plt.subplots(1, 3, figsize=(22, 5))
    axes[0].bar(plot_df['label'], plot_df['avg_quality'], color='#2a6f97')
    axes[0].set_title('Average Benchmark Quality')
    axes[0].tick_params(axis='x', rotation=90)
    axes[1].bar(plot_df['label'], plot_df['avg_latency_seconds'], color='#c1121f')
    axes[1].set_title('Average Latency (s)')
    axes[1].tick_params(axis='x', rotation=90)
    axes[2].bar(plot_df['label'], plot_df['peak_gpu_memory_mb'], color='#6a994e')
    axes[2].set_title('Peak GPU Memory (MB)')
    axes[2].tick_params(axis='x', rotation=90)
    plt.tight_layout()
    plt.show()
else:
    print('Leaderboard not ready yet.')


In [ ]:
# Simple recommendation table.
import pandas as pd

if 'leaderboard' in globals() and not leaderboard.empty:
    best_quality = leaderboard.sort_values(['avg_quality', 'avg_latency_seconds'], ascending=[False, True]).iloc[0]
    best_latency = leaderboard.sort_values('avg_latency_seconds', ascending=True).iloc[0]
    best_memory = leaderboard.dropna(subset=['peak_gpu_memory_mb']).sort_values('peak_gpu_memory_mb', ascending=True).iloc[0]
    recommendations = pd.DataFrame([
        {'category': 'best_for_quality', 'scenario_name': best_quality['scenario_name'], 'method': best_quality['method'], 'budget': best_quality['budget']},
        {'category': 'best_for_latency', 'scenario_name': best_latency['scenario_name'], 'method': best_latency['method'], 'budget': best_latency['budget']},
        {'category': 'best_for_memory', 'scenario_name': best_memory['scenario_name'], 'method': best_memory['method'], 'budget': best_memory['budget']},
    ])
    display(recommendations)
else:
    print('Leaderboard not ready yet.')


## Granular Task-Level Analysis

This section focuses on dissecting the `metrics.json` files to extract individual task performance, allowing for a deeper understanding of how different KV Cache strategies perform on specific sub-tasks.

In [ ]:
import json
import pandas as pd
from pathlib import Path

if 'combined_summary' in globals() and not combined_summary.empty:
    task_metrics_data = []

    for index, row in combined_summary.iterrows():
        metrics_path = Path(row['metrics_path'])
        if metrics_path.exists():
            with open(metrics_path, 'r', encoding='utf-8') as f:
                metrics = json.load(f)

            # Extract task-specific metrics
            for task_prefix, task_results in metrics.items():
                # Flatten task_results if there are multiple metrics per task (e.g., {'string_match': 0.8, 'exact_match': 0.7})
                for metric_name, metric_value in task_results.items():
                    task_metrics_data.append({
                        'scenario_name': row['scenario_name'],
                        'dataset': row['dataset'],
                        'data_dir': row['data_dir'],
                        'task_prefix': task_prefix, # Original task prefix like 'niah', 'qa', 'cwe', 'fwe'
                        'method': row['method'],
                        'budget': row['budget'],
                        'metric_name': metric_name,
                        'metric_value': metric_value
                    })

    if task_metrics_data:
        granular_df = pd.DataFrame(task_metrics_data)
        print("### Head of Granular Task-Level Metrics DataFrame:")
        display(granular_df.head())

        print("\n### Unique Metrics Found:")
        print(granular_df['metric_name'].unique())

        print("\n### Unique Task Prefixes Found:")
        print(granular_df['task_prefix'].unique())

    else:
        print('No task-level metrics could be loaded.')
else:
    print('Combined summary not available. Please run the aggregation cells first.')

### Next Steps for Granular Analysis:

Now that we have the `granular_df` DataFrame containing task-level metrics, we can:

1.  **Filter and Group**: Select specific `task_prefix` values (e.g., 'niah', 'qa') and group by `method` to compare their performance on those specific tasks.
2.  **Visualize**: Create plots (e.g., bar plots, box plots) to visualize the `metric_value` for different methods on a given `task_prefix`.
3.  **Identify Strengths**: Look for methods that consistently perform better (higher `metric_value`) on certain `task_prefix` types. This will help in formulating recommendations.

### Visualizing Granular Task Performance

Below are comparative plots for each unique task prefix found in the data, showing the `metric_value` (e.g., string match score) for each KV Cache method. This detailed view helps in pinpointing which strategies excel in particular sub-tasks.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'granular_df' in globals() and not granular_df.empty:
    unique_task_prefixes = granular_df['task_prefix'].unique()
    unique_metrics = granular_df['metric_name'].unique()

    # Iterate through each unique task prefix
    for task_prefix in unique_task_prefixes:
        print(f"### Analyzing Task Prefix: {task_prefix}")
        task_prefix_df = granular_df[granular_df['task_prefix'] == task_prefix].copy()

        # Iterate through each unique metric for this task prefix
        for metric in unique_metrics:
            metric_df = task_prefix_df[task_prefix_df['metric_name'] == metric].copy()

            if not metric_df.empty:
                plt.figure(figsize=(10, 6))
                sns.barplot(x='method', y='metric_value', data=metric_df,
                            palette='viridis', hue='method', legend=False)
                plt.title(f'Performance for {task_prefix} - Metric: {metric}')
                plt.xlabel('KV Cache Method')
                plt.ylabel(f'{metric} Value')
                plt.xticks(rotation=45, ha='right')
                plt.tight_layout()
                plt.show()
            else:
                print(f"No data for metric '{metric}' in task prefix '{task_prefix}'.")
        print("\n" + "-"*100 + "\n") # Separator for readability

else:
    print('Granular DataFrame not available. Please ensure the previous cells generating granular_df have been run.')

### Aggregated Analysis for Main Task Categories

This section provides an aggregated view of the performance for each KV Cache method across the four main task categories: Needle In A Haystack, Question Answering, Multi-Hop Tracing, and Aggregation.

In [ ]:
import numpy as np

if 'granular_df' in globals() and not granular_df.empty:
    # Define mapping from granular task prefixes to main task categories
    task_category_map = {
        'niah': 'Needle In A Haystack',
        'qa': 'Question Answering',
        'vt': 'Multi-Hop Tracing',
        'cwe': 'Aggregation',
        'fwe': 'Aggregation',
    }

    # Function to apply the mapping (handles variations like 'niah_multikey_1')
    def get_main_task_category(task_prefix):
        for key, category in task_category_map.items():
            if task_prefix.startswith(key):
                return category
        return 'Other' # Fallback for any unmapped prefixes

    # Apply the mapping to create a new 'main_task_category' column
    granular_df['main_task_category'] = granular_df['task_prefix'].apply(get_main_task_category)

    # Aggregate by main_task_category, method, and metric_name
    aggregated_main_tasks_df = granular_df.groupby(['main_task_category', 'method', 'metric_name'])['metric_value'].mean().reset_index()

    print("### Aggregated Metrics per Main Task Category:")
    display(aggregated_main_tasks_df.head())

else:
    print('Granular DataFrame not available. Please ensure the previous cells generating granular_df have been run.')

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'aggregated_main_tasks_df' in globals() and not aggregated_main_tasks_df.empty:
    unique_main_task_categories = aggregated_main_tasks_df['main_task_category'].unique()
    unique_metrics = aggregated_main_tasks_df['metric_name'].unique()

    for main_category in unique_main_task_categories:
        print(f"### Visualizing Main Task Category: {main_category}")
        category_df = aggregated_main_tasks_df[aggregated_main_tasks_df['main_task_category'] == main_category].copy()

        for metric in unique_metrics:
            metric_category_df = category_df[category_df['metric_name'] == metric].copy()

            if not metric_category_df.empty:
                plt.figure(figsize=(12, 7))
                sns.barplot(x='method', y='metric_value', data=metric_category_df,
                            palette='viridis', hue='method')
                plt.title(f'Average Performance for {main_category} - Metric: {metric}')
                plt.xlabel('KV Cache Method')
                plt.ylabel(f'Average {metric} Value')
                plt.xticks(rotation=45, ha='right')
                plt.legend(title='Method', bbox_to_anchor=(1.05, 1), loc='upper left')
                plt.tight_layout()
                plt.show()
            else:
                print(f"No data for metric '{metric}' in main task category '{main_category}'.")
        print("\n" + "-"*100 + "\n") # Separator for readability
else:
    print('Aggregated main tasks DataFrame not available. Please ensure the previous cells have been run.')